# 🚀 CAVAT - Remote Dev Environment Setup

**Mục đích:** Thiết lập kết nối SSH từ VS Code (Local) tới Kaggle GPU thông qua Ngrok Tunnel.

**Dự án:** CAVAT - Hệ thống AI Trợ giảng Toán học (Qwen 2.5 + QLoRA + Custom RAG)

---

### ⚡ Hướng dẫn sử dụng:
1. Đảm bảo **GPU T4** đã bật (Settings → Accelerator → GPU T4 x2)
2. Bật **Internet** (Settings → Internet → On)
3. Chạy các cell **tuần tự** từ trên xuống dưới
4. Sau Cell 3, copy lệnh SSH hiển thị và dùng để kết nối từ VS Code

## 📋 Cell 1: Kiểm tra GPU & Môi trường

In [ ]:
# ============================================
# CELL 1: KIỂM TRA GPU & MÔI TRƯỜNG HỆ THỐNG
# ============================================

import torch
import subprocess
import sys
import os

print("="*60)
print("🔍 KIỂM TRA MÔI TRƯỜNG HỆ THỐNG CAVAT")
print("="*60)

# Kiểm tra GPU
print("\n📊 [GPU STATUS]")
!nvidia-smi

print(f"\n🔧 [PYTORCH INFO]")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   GPU Name:        {gpu_name}")
    print(f"   GPU Memory:      {gpu_mem:.1f} GB")
    print(f"   CUDA Version:    {torch.version.cuda}")
    print(f"\n   ✅ GPU sẵn sàng cho training Qwen 2.5 + QLoRA!")
else:
    print("\n   ❌ KHÔNG TÌM THẤY GPU!")
    print("   → Vào Settings → Accelerator → Chọn GPU T4 x2")
    print("   → Restart notebook sau khi thay đổi")

# Kiểm tra Python
print(f"\n🐍 [PYTHON INFO]")
print(f"   Python version:  {sys.version}")
print(f"   Working dir:     {os.getcwd()}")
print(f"   Disk space:")
!df -h /kaggle/working | tail -1

print("\n" + "="*60)
print("✅ Kiểm tra hoàn tất - Tiếp tục Cell 2")
print("="*60)

## 🔧 Cell 2: Cài đặt SSH Server + Ngrok

In [ ]:
# ============================================
# CELL 2: CÀI ĐẶT SSH SERVER + PYNGROK
# ============================================

import os

print("="*60)
print("🔧 CÀI ĐẶT SSH SERVER & NGROK")
print("="*60)

# --- Bước 2.1: Cài đặt OpenSSH Server ---
print("\n📦 [1/4] Cài đặt OpenSSH Server...")
!apt-get update -qq && apt-get install -y -qq openssh-server > /dev/null 2>&1
print("   ✅ OpenSSH Server đã cài đặt")

# --- Bước 2.2: Cấu hình SSH ---
print("\n🔐 [2/4] Cấu hình SSH Server...")
!mkdir -p /var/run/sshd

# Đặt mật khẩu root
!echo 'root:cavat123' | chpasswd

# Cho phép root login qua SSH
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config

# Cho phép password authentication
!sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config
!sed -i 's/PasswordAuthentication no/PasswordAuthentication yes/' /etc/ssh/sshd_config

print("   ✅ SSH đã cấu hình (Root login: YES, Password auth: YES)")

# --- Bước 2.3: Khởi chạy SSH ---
print("\n🚀 [3/4] Khởi chạy SSH Server...")
!service ssh start
print("   ✅ SSH Server đang chạy trên port 22")

# --- Bước 2.4: Cài pyngrok (Python wrapper cho Ngrok) ---
print("\n🌐 [4/4] Cài đặt pyngrok...")
!pip install pyngrok -q
print("   ✅ pyngrok đã cài đặt")

print("\n" + "="*60)
print("✅ Cài đặt hoàn tất - Tiếp tục Cell 3")
print("="*60)

## 🌐 Cell 3: Khởi chạy Ngrok Tunnel & Hiển thị Endpoint

⚠️ **QUAN TRỌNG:** Thay `NGROK_TOKEN` bên dưới bằng token của bạn từ https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ============================================
# CELL 3: KHỞI CHẠY NGROK TUNNEL (via pyngrok)
# ============================================

from pyngrok import ngrok
import json
import os

print("="*60)
print("🌐 KHỞI CHẠY NGROK SSH TUNNEL")
print("="*60)

# ┌──────────────────────────────────────────────────────┐
# │  ⚠️  THAY TOKEN CỦA BẠN VÀO ĐÂY                   │
# │  Lấy tại: dashboard.ngrok.com/get-started/your-authtoken │
# └──────────────────────────────────────────────────────┘
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"

# --- Kiểm tra token ---
if NGROK_TOKEN == "PASTE_YOUR_TOKEN_HERE":
    print("\n   ❌ BẠN CHƯA NHẬP TOKEN!")
    print("   → Vào https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   → Copy token và paste vào biến NGROK_TOKEN ở trên")
    print("   → Chạy lại cell này")
    raise SystemExit("Token chưa được cấu hình")

# --- Bước 3.1: Đăng ký token ---
print("\n🔑 [1/3] Đăng ký Ngrok AuthToken...")
ngrok.set_auth_token(NGROK_TOKEN)
print("   ✅ Token đã đăng ký thành công")

# --- Bước 3.2: Kill tunnel cũ (nếu có) ---
print("\n🔄 [2/3] Dọn dẹp tunnel cũ...")
ngrok.kill()
print("   ✅ Đã dọn dẹp")

# --- Bước 3.3: Mở tunnel ---
print("\n🚇 [3/3] Mở SSH Tunnel...")
try:
    tunnel = ngrok.connect(22, "tcp")
    public_url = tunnel.public_url  # tcp://x.tcp.ngrok.io:yyyyy
    
    # Parse host:port
    addr = public_url.replace("tcp://", "")
    host, port = addr.rsplit(":", 1)
    
    print("\n" + "🟢"*30)
    print("\n" + "="*60)
    print("   🎉 NGROK TUNNEL ĐÃ SẴN SÀNG!")
    print("="*60)
    print(f"")
    print(f"   🔗 Host:     {host}")
    print(f"   🔗 Port:     {port}")
    print(f"   🔑 Password: cavat123")
    print(f"")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  📋 LỆNH KẾT NỐI (copy vào Terminal):          │")
    print(f"   │                                                  │")
    print(f"   │  ssh root@{host} -p {port}")
    print(f"   │                                                  │")
    print(f"   └──────────────────────────────────────────────────┘")
    print(f"")
    print(f"   📝 VS Code SSH Config (~/.ssh/config):")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  Host kaggle-cavat                               │")
    print(f"   │      HostName {host}")
    print(f"   │      Port {port}")
    print(f"   │      User root                                   │")
    print(f"   │      StrictHostKeyChecking no                    │")
    print(f"   │      UserKnownHostsFile /dev/null                │")
    print(f"   └──────────────────────────────────────────────────┘")
    print("\n" + "🟢"*30)
    
    # Lưu endpoint vào file để các cell khác sử dụng
    with open('/tmp/ngrok_endpoint.json', 'w') as f:
        json.dump({'host': host, 'port': port, 'url': public_url}, f)
    
except Exception as e:
    print(f"\n   ❌ LỖI KHI TẠO TUNNEL!")
    print(f"")
    print(f"   Chi tiết lỗi: {e}")
    print(f"")
    print(f"   🔍 Cách khắc phục:")
    print(f"   1. Kiểm tra token đúng chưa → dashboard.ngrok.com/get-started/your-authtoken")
    print(f"   2. Kiểm tra Internet đã bật → Settings → Internet → On")
    print(f"   3. Free tier hết session → Đợi vài phút rồi chạy lại Cell 3")
    print(f"   4. Token bị lỗi định dạng → Copy lại nguyên token, không thêm/bớt ký tự")

## 🐕 Cell 4: Watchdog Heartbeat (Giám sát tự động)

In [ ]:
# ============================================
# CELL 4: WATCHDOG HEARTBEAT - GIÁM SÁT TỰ ĐỘNG
# ============================================

import threading
import time
import os
from pyngrok import ngrok
from datetime import datetime

print("="*60)
print("🐕 KHỞI CHẠY WATCHDOG HEARTBEAT")
print("="*60)

# Cấu hình
HEARTBEAT_INTERVAL = 60  # Kiểm tra mỗi 60 giây
MAX_SSH_RESTARTS = 5     # Tối đa 5 lần restart SSH

class WatchdogMonitor:
    def __init__(self):
        self.running = True
        self.ssh_restarts = 0
        self.check_count = 0
        self.last_ngrok_ok = True
        
    def check_ssh(self):
        """Kiểm tra SSH server còn chạy không"""
        try:
            result = os.popen("service ssh status 2>&1").read()
            return "running" in result.lower() or "active" in result.lower()
        except:
            return False
    
    def check_ngrok(self):
        """Kiểm tra Ngrok tunnel còn hoạt động không"""
        try:
            tunnels = ngrok.get_tunnels()
            return len(tunnels) > 0
        except:
            return False
    
    def restart_ssh(self):
        """Khởi động lại SSH server"""
        os.system("service ssh start")
        self.ssh_restarts += 1
    
    def heartbeat(self):
        """Vòng lặp kiểm tra chính"""
        while self.running:
            self.check_count += 1
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            ssh_ok = self.check_ssh()
            ngrok_ok = self.check_ngrok()
            
            # Status icons
            ssh_icon = "✅" if ssh_ok else "❌"
            ngrok_icon = "✅" if ngrok_ok else "❌"
            
            if ssh_ok and ngrok_ok:
                # In status mỗi 5 lần check để không spam
                if self.check_count % 5 == 0:
                    print(f"[{timestamp}] #{self.check_count:04d} | SSH: {ssh_icon} | Ngrok: {ngrok_icon} | Uptime: {self.check_count} min")
            else:
                # Luôn in khi có lỗi
                print(f"[{timestamp}] #{self.check_count:04d} | SSH: {ssh_icon} | Ngrok: {ngrok_icon} | ⚠️ CẢNH BÁO!")
                
                if not ssh_ok:
                    if self.ssh_restarts < MAX_SSH_RESTARTS:
                        print(f"   🔄 SSH DOWN → Đang restart (lần {self.ssh_restarts + 1}/{MAX_SSH_RESTARTS})...")
                        self.restart_ssh()
                    else:
                        print(f"   🛑 SSH đã restart {MAX_SSH_RESTARTS} lần, cần kiểm tra thủ công!")
                
                if not ngrok_ok:
                    if self.last_ngrok_ok:  # Chỉ cảnh báo lần đầu mất kết nối
                        print(f"   🚨 NGROK TUNNEL ĐÃ MẤT KẾT NỐI!")
                        print(f"   → Chạy lại Cell 3 để tạo tunnel mới")
                        print(f"   → Sau đó cập nhật SSH config với endpoint mới")
            
            self.last_ngrok_ok = ngrok_ok
            time.sleep(HEARTBEAT_INTERVAL)

# Khởi chạy watchdog
watchdog = WatchdogMonitor()
watchdog_thread = threading.Thread(target=watchdog.heartbeat, daemon=True)
watchdog_thread.start()

print(f"\n🐕 Watchdog đã khởi chạy thành công!")
print(f"   • Interval:      {HEARTBEAT_INTERVAL}s")
print(f"   • Max SSH retry: {MAX_SSH_RESTARTS}")
print(f"   • Status log:    Mỗi 5 phút (im lặng khi ổn định)")
print(f"   • Alert:         Ngay lập tức khi phát hiện lỗi")
print(f"\n   ℹ️  Để dừng watchdog: watchdog.running = False")
print("\n" + "="*60)
print("✅ Watchdog đang giám sát - Tiếp tục Cell 5")
print("="*60)

## 📁 Cell 5: Clone Project & Thiết lập Git Auto-save

In [ ]:
# ============================================
# CELL 5: CLONE PROJECT & GIT AUTO-SAVE
# ============================================

import os
import time

print("="*60)
print("📁 THIẾT LẬP PROJECT & GIT AUTO-SAVE")
print("="*60)

# ┌─────────────────────────────────────────────────┐
# │           CẤU HÌNH GIT (ĐÃ CẤU HÌNH)          │
# └─────────────────────────────────────────────────┘
GIT_REPO = "https://github.com/minhvuogdzz/MathAI-support-Project.git"
GIT_USER = "minhvuogdzz"
GIT_EMAIL = "minhvuogdzz@users.noreply.github.com"
PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

# --- Bước 5.1: Cấu hình Git toàn cục ---
print("\n🔧 [1/3] Cấu hình Git...")
!git config --global user.name "{GIT_USER}"
!git config --global user.email "{GIT_EMAIL}"
!git config --global init.defaultBranch main
print(f"   ✅ Git user: {GIT_USER}")

# --- Bước 5.2: Clone hoặc Init project ---
print(f"\n📦 [2/3] Thiết lập project tại {PROJECT_DIR}...")

if os.path.exists(PROJECT_DIR):
    print(f"   ℹ️  Thư mục đã tồn tại, pull latest...")
    os.chdir(PROJECT_DIR)
    !git pull origin main 2>/dev/null || echo "   ℹ️  Không có thay đổi mới"
else:
    # Thử clone trước
    clone_result = os.system(f"git clone {GIT_REPO} {PROJECT_DIR} 2>/dev/null")
    if clone_result != 0:
        # Repo trống, tạo mới và kết nối
        print("   ℹ️  Repo trống, khởi tạo local và kết nối remote...")
        os.makedirs(PROJECT_DIR, exist_ok=True)
        os.chdir(PROJECT_DIR)
        !git init
        !git remote add origin {GIT_REPO}
    else:
        os.chdir(PROJECT_DIR)

print(f"   ✅ Project directory: {PROJECT_DIR}")

# --- Bước 5.3: Hiển thị cấu trúc ---
print(f"\n📂 [3/3] Cấu trúc project hiện tại:")
!find . -maxdepth 2 -not -path './.git/*' -not -name '.git' | head -30

print("\n" + "="*60)
print("✅ Project đã sẵn sàng - Tiếp tục Cell 6")
print("="*60)

## 💾 Cell 6: Auto-save Functions (Gọi trước khi đóng session)

In [ ]:
# ============================================
# CELL 6: AUTO-SAVE FUNCTIONS
# ============================================

import os
import time
import threading
from datetime import datetime

PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

def save_now(message=None):
    """
    💾 Lưu toàn bộ thay đổi lên GitHub ngay lập tức.
    
    Sử dụng:
        save_now()                           # Auto message
        save_now("Hoàn thành training v1")   # Custom message
    """
    os.chdir(PROJECT_DIR)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if message is None:
        message = f"Auto-save session {timestamp}"
    else:
        message = f"{message} [{timestamp}]"
    
    print(f"\n💾 Đang lưu: {message}")
    
    # Stage all changes
    os.system("git add -A")
    
    # Check if there are changes to commit
    status = os.popen("git status --porcelain").read().strip()
    if not status:
        staged = os.popen("git diff --cached --name-only").read().strip()
        if not staged:
            print("   ℹ️  Không có thay đổi mới để lưu.")
            return
    
    # Commit
    commit_result = os.system(f'git commit -m "{message}"')
    if commit_result != 0:
        print("   ℹ️  Không có thay đổi mới để commit.")
        return
    
    # Push
    push_result = os.system("git push -u origin main")
    if push_result == 0:
        print(f"   ✅ Đã push thành công lên GitHub!")
    else:
        print(f"   ❌ Push thất bại! Kiểm tra:")
        print(f"      1. GitHub token/credentials đã cấu hình?")
        print(f"      2. Repo URL đúng không?")
        print(f"      → Thử: git remote set-url origin https://<TOKEN>@github.com/minhvuogdzz/MathAI-support-Project.git")


def auto_save_periodic(interval_minutes=30):
    """
    ⏰ Tự động lưu định kỳ.
    
    Sử dụng:
        auto_save_periodic(30)  # Lưu mỗi 30 phút
    """
    def _auto_save_loop():
        count = 0
        while True:
            time.sleep(interval_minutes * 60)
            count += 1
            print(f"\n⏰ [Auto-save #{count}]")
            save_now(f"Periodic auto-save #{count}")
    
    thread = threading.Thread(target=_auto_save_loop, daemon=True)
    thread.start()
    print(f"⏰ Auto-save đã bật: lưu mỗi {interval_minutes} phút")


# Hiển thị hướng dẫn
print("="*60)
print("💾 AUTO-SAVE FUNCTIONS ĐÃ SẴN SÀNG")
print("="*60)
print(f"")
print(f"   📋 Các lệnh có sẵn:")
print(f"")
print(f"   save_now()                      → Lưu ngay lập tức")
print(f"   save_now('message')             → Lưu với ghi chú")
print(f"   auto_save_periodic(30)          → Tự động lưu mỗi 30 phút")
print(f"")
print(f"   ⚠️  QUAN TRỌNG: Luôn gọi save_now() trước khi đóng session!")
print(f"")
print("="*60)
print("\n🎉 SETUP HOÀN TẤT! Bạn có thể kết nối VS Code ngay bây giờ.")
print("="*60)

---

## 📌 Quick Reference

| Hành động | Lệnh |
|-----------|-------|
| Kết nối SSH | `ssh root@<HOST> -p <PORT>` |
| Mật khẩu | `cavat123` |
| Lưu code | `save_now()` hoặc `save_now('message')` |
| Auto-save | `auto_save_periodic(30)` |
| Kiểm tra GPU | `nvidia-smi` |
| Dừng watchdog | `watchdog.running = False` |
| Restart tunnel | Chạy lại Cell 3 |